# 03 — Comparación de algoritmos (Benchmark 3D-BPP)

> **Catálogo homogéneo `showcase_master_catalog.json`** — mismos SKUs P1–P5 en 3D-BPP, Container Loading y Single Container. Cada tipo usa una **instancia showcase** compleja donde el benchmark **diferencia** los algoritmos del grupo.

**Instancia:** `showcase_3d_bpp_instance.json`

## Flujo de este notebook

1. **Problema** — qué hay que empacar y bajo qué reglas
2. **Algoritmos** — 5 heurísticas internas + adaptador `py3dbp` (si está instalado)
3. **Ejecución** — misma instancia, mismo validador, mismas métricas
4. **Resultados** — quién empaca más y quién aprovecha mejor el espacio
5. **Layouts** — comparación visual del mejor vs el peor

En esta instancia se espera diferenciación clara: `best_fit` ~27 empacados vs `extreme_points` ~22.

Cada motor del benchmark también es invocable individualmente vía `POST /algorithms/{name}/execute`.


In [ ]:
# Configuración del entorno (ejecutar primero)
%matplotlib inline

import sys
from pathlib import Path
import matplotlib.pyplot as plt

_cwd = Path.cwd().resolve()
for _candidate in [_cwd, *_cwd.parents]:
    _nb = _candidate / "notebooks"
    if (_nb / "_shared" / "loaders.py").exists():
        if str(_nb) not in sys.path:
            sys.path.insert(0, str(_nb))
        break
else:
    raise RuntimeError("No se encontró notebooks/_shared. Abre el notebook desde packing-services/.")

from _shared.loaders import setup_paths
ROOT = setup_paths(_cwd)
from _shared import display, viz, runners
from _shared import viz_3d

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print(f"Proyecto: {ROOT}")


In [ ]:
from _shared.loaders import load_showcase_instance

instance = load_showcase_instance('3D_BPP')
print('═══ 1. PROBLEMA A RESOLVER ═══')
display.show_problem_overview(instance)


In [ ]:
print('═══ BENCHMARK HOMOGÉNEO (3D-BPP) ═══')
response = display.run_and_show_showcase_benchmark('3D_BPP', instance)
print(f'Ganador: {response.ranking[0]}')


In [ ]:
print('═══ 5. LAYOUTS: MEJOR vs PEOR ═══')
from packing_services.domain.models import Container
containers = [Container(**c) for c in instance['containers']]
best = next(r for r in response.results if r.engine == response.ranking[0])
worst = next(r for r in response.results if r.engine == response.ranking[-1])
for label, result in [('MEJOR', best), ('PEOR', worst)]:
    if result.solution:
        print(f'--- {label}: {result.engine} ({result.metrics.items_packed} empacados, {result.metrics.volume_utilization*100:.1f}%) ---')
        fig = viz.plot_all_container_views(result.solution, containers, title_prefix=label)
        if fig: plt.show()
        for fig3d in viz_3d.plot_solution_3d(result.solution, containers, title_prefix=f'{label} 3D'):
            fig3d.show()


**Mensaje clave:** misma entrada, distintas salidas — el benchmark cuantifica la brecha entre heurísticas.

**Siguiente:** mejora local y otros tipos en `08_catalogo_algoritmos_y_futuro.ipynb`.

Luego: `04_validacion_independiente.ipynb`
